# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhruvaiyer2-netizen/FlyRankMLProject/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The source for the data from the table is page-per-day, but for my decision unit, I intend to aggregate it into as page-per-month.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

impressions, clicks, ctr, avg_position/position, position_tier, sessions, engagement_rate, scroll_rate, word_count, content type, main_intent, age/freshness are all in the features bucket.
ga4_data_available, report_date, client_hash_id, content_hash_id, keyword_hash_id, url_hash_id, gsc_data_start, ga4_data_start are all in the context bucket.
tier-relative ctr gap (will be derived by me) belongs in the label/proxy bucket.
health_score, priority_score, action_type, needs_ctr_fix, is_quick_win, refresh_tier are generated by flyrank, so feeding this to a model would just teach it to copy the product instead of actually learning.
raw query, urls, title domain text, client text are all scrambled before release so they do not bias or influence the training of the model, not to mention they are confidential pieces of data.
AI Session fields (sessions_ai) is too sparse to support the chosen lane.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
duckdb.sql("""
    SELECT
      COUNT(*) AS total_rows,
      COUNT(DISTINCT report_date || client_hash_id || content_hash_id) AS unique_keys
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    WHERE month = '2026-03'
""").show()
duckdb.sql("""
    SELECT
      COUNT(*) AS n_rows,
      MIN(report_date) AS earliest,
      MAX(report_date) AS latest,
      COUNT(DISTINCT client_hash_id) AS n_clients,
      COUNT(DISTINCT content_hash_id) AS n_pages
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    WHERE month = '2026-03'
""").show()
duckdb.sql("""
    SELECT
      COUNT(*) AS total_march_rows,
      COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM march_facts
""").show()
duckdb.sql("""
    SELECT
      COUNT(DISTINCT client_hash_id) AS total_clients,
      COUNT(DISTINCT CASE WHEN ga4_data_available THEN client_hash_id END) AS clients_with_ga4
    FROM march_facts
""").show()

┌───────────────┬──────────────────┐
│ total_clients │ clients_with_ga4 │
│     int64     │      int64       │
├───────────────┼──────────────────┤
│            55 │               41 │
└───────────────┴──────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can never tell me a single page's typical behaviour across clients, because history length is unbalanced across the 55 clients in this slice, nor can it tell me about enagement or session behaviour for all the rows with ga4_data_available = False. It also cannot tell me any future prediction claim if the feature and label window share days - that would leak the answer into the input rather than test whether or not the model can find it independently. All three points mentioned above are structural limits of the data rather than somethingt that can be fixed by better models or more rows.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.